# Gold Layer

This notebook creates the final analysis output from the cleaned Silver dataset.

In [0]:
#Import Libraries

import pyspark.sql.functions as F
from pyspark.sql.types import *




In [0]:
#Define Catalog Name
catalog_name = "carnot-catalog"

In [0]:
#Read Silver Table

parcel_timeseries_silver = spark.table(
    f"`{catalog_name}`.silver.silver_parcel_timeseries"
)

display(parcel_timeseries_silver)

parcel_id,reading_date,ndvi_value,temperature_c,rainfall_mm,sensor_status,mill_id,crop_type,sowing_date,area_hectares
PARCEL_015,2026-01-11,0.257,34.0,0.0,ok,MILL_SOUTH,sugarcane,2026-01-06,4.87
PARCEL_014,2026-03-18,0.769,18.0,7.8,ok,MILL_NORTH,sugarcane,2026-01-05,9.39
PARCEL_014,2026-01-07,0.288,19.4,0.0,ok,MILL_NORTH,sugarcane,2026-01-05,9.39
PARCEL_015,2026-02-13,0.802,16.3,0.0,ok,MILL_SOUTH,sugarcane,2026-01-06,4.87
PARCEL_019,2026-01-30,0.335,24.7,0.0,ok,MILL_SOUTH,sugarcane,2026-01-18,10.19
PARCEL_011,2026-04-01,0.586,24.8,0.0,ok,MILL_NORTH,wheat,2026-02-21,6.67
PARCEL_004,2026-02-02,0.734,17.9,0.0,ok,MILL_NORTH,sugarcane,2026-01-02,3.18
PARCEL_025,2026-02-23,0.493,23.8,0.0,ok,MILL_WEST,sugarcane,2026-02-07,3.02
PARCEL_007,2026-04-09,0.696,25.6,0.0,ok,MILL_NORTH,sugarcane,2026-01-18,8.53
PARCEL_023,2026-03-31,0.525,23.8,0.0,ok,MILL_WEST,sugarcane,2026-01-26,3.55


Ignore rows where sensor_status is bad

In [0]:
#shows unique sensor status values

parcel_timeseries_silver.select("sensor_status").distinct().display()

sensor_status
ok
na
error
nan
null


In [0]:
#Ignore rows where sensor_status is bad

gold_df = parcel_timeseries_silver.filter(
    F.col("sensor_status") == "ok"
)

In [0]:
#display(gold_df)

In [0]:
## Calculate difference between reading date and sowing date


gold_df = gold_df.withColumn(
    "days_from_sowing",
    F.datediff(
        F.col("reading_date"),
        F.col("sowing_date")
    )
)

In [0]:
#display(gold_df)

parcel_id,reading_date,ndvi_value,temperature_c,rainfall_mm,sensor_status,mill_id,crop_type,sowing_date,area_hectares,days_from_sowing
PARCEL_015,2026-01-11,0.257,34.0,0.0,ok,MILL_SOUTH,sugarcane,2026-01-06,4.87,5
PARCEL_014,2026-03-18,0.769,18.0,7.8,ok,MILL_NORTH,sugarcane,2026-01-05,9.39,72
PARCEL_014,2026-01-07,0.288,19.4,0.0,ok,MILL_NORTH,sugarcane,2026-01-05,9.39,2
PARCEL_015,2026-02-13,0.802,16.3,0.0,ok,MILL_SOUTH,sugarcane,2026-01-06,4.87,38
PARCEL_019,2026-01-30,0.335,24.7,0.0,ok,MILL_SOUTH,sugarcane,2026-01-18,10.19,12
PARCEL_011,2026-04-01,0.586,24.8,0.0,ok,MILL_NORTH,wheat,2026-02-21,6.67,39
PARCEL_004,2026-02-02,0.734,17.9,0.0,ok,MILL_NORTH,sugarcane,2026-01-02,3.18,31
PARCEL_025,2026-02-23,0.493,23.8,0.0,ok,MILL_WEST,sugarcane,2026-02-07,3.02,16
PARCEL_007,2026-04-09,0.696,25.6,0.0,ok,MILL_NORTH,sugarcane,2026-01-18,8.53,81
PARCEL_023,2026-03-31,0.525,23.8,0.0,ok,MILL_WEST,sugarcane,2026-01-26,3.55,64


In [0]:
# # Readings from 30 days before sowing


before_df = gold_df.filter(
    (F.col("days_from_sowing") >= -30) &
    (F.col("days_from_sowing") < 0)
)

In [0]:
#display(before_df)

parcel_id,reading_date,ndvi_value,temperature_c,rainfall_mm,sensor_status,mill_id,crop_type,sowing_date,area_hectares,days_from_sowing
PARCEL_011,2026-02-06,0.166,33.1,0.0,ok,MILL_NORTH,wheat,2026-02-21,6.67,-15
PARCEL_024,2026-01-07,0.188,25.6,3.0,ok,MILL_EAST,soybean,2026-01-16,5.43,-9
PARCEL_016,2026-01-02,0.144,19.2,0.0,ok,MILL_EAST,sugarcane,2026-01-30,4.76,-28
PARCEL_012,2026-02-24,0.179,26.5,0.0,ok,MILL_WEST,sugarcane,2026-03-01,6.85,-5
PARCEL_011,2026-02-08,0.185,31.2,0.0,ok,MILL_NORTH,wheat,2026-02-21,6.67,-13
PARCEL_008,2026-01-10,0.231,22.6,0.0,ok,MILL_EAST,sugarcane,2026-01-22,2.98,-12
PARCEL_003,2026-02-12,0.241,23.7,0.0,ok,MILL_NORTH,soybean,2026-02-13,5.35,-1
PARCEL_006,2026-01-14,0.046,20.7,0.0,ok,MILL_WEST,sugarcane,2026-02-11,5.67,-28
PARCEL_007,2026-01-05,0.156,25.6,0.0,ok,MILL_NORTH,sugarcane,2026-01-18,8.53,-13
PARCEL_018,2026-01-07,0.041,19.5,0.0,ok,MILL_SOUTH,sugarcane,2026-01-11,5.82,-4


In [0]:
# Readings from 30 days after sowing

after_df = gold_df.filter(
    (F.col("days_from_sowing") > 0) &
    (F.col("days_from_sowing") <= 30)
)

In [0]:
#display(after_df)

parcel_id,reading_date,ndvi_value,temperature_c,rainfall_mm,sensor_status,mill_id,crop_type,sowing_date,area_hectares,days_from_sowing
PARCEL_015,2026-01-11,0.257,34.0,0.0,ok,MILL_SOUTH,sugarcane,2026-01-06,4.87,5
PARCEL_014,2026-01-07,0.288,19.4,0.0,ok,MILL_NORTH,sugarcane,2026-01-05,9.39,2
PARCEL_019,2026-01-30,0.335,24.7,0.0,ok,MILL_SOUTH,sugarcane,2026-01-18,10.19,12
PARCEL_025,2026-02-23,0.493,23.8,0.0,ok,MILL_WEST,sugarcane,2026-02-07,3.02,16
PARCEL_010,2026-02-01,0.208,18.1,0.0,ok,MILL_EAST,sugarcane,2026-01-07,7.44,25
PARCEL_010,2026-01-15,0.375,17.0,0.0,ok,MILL_EAST,sugarcane,2026-01-07,7.44,8
PARCEL_010,2026-01-22,0.339,34.0,0.0,ok,MILL_EAST,sugarcane,2026-01-07,7.44,15
PARCEL_002,2026-02-05,0.43,32.3,0.0,ok,MILL_SOUTH,sugarcane,2026-01-16,8.97,20
PARCEL_019,2026-01-24,0.183,24.0,3.7,ok,MILL_SOUTH,sugarcane,2026-01-18,10.19,6
PARCEL_006,2026-02-23,0.468,34.1,0.0,ok,MILL_WEST,sugarcane,2026-02-11,5.67,12


In [0]:
# Mean ndvi before sowing by crop type
before_agg = before_df.groupBy("crop_type").agg(
    F.avg("ndvi_value").alias("mean_ndvi_before")
)

In [0]:
#display(before_agg)

crop_type,mean_ndvi_before
wheat,0.1761372534959924
soybean,0.17062195180347417
sugarcane,0.17748928531073033


In [0]:
# Mean ndvi after sowing and parcel counts
after_agg = after_df.groupBy("crop_type").agg(
    F.avg("ndvi_value").alias("mean_ndvi_after"),
    F.countDistinct("parcel_id").alias("n_parcels")

)

In [0]:
#display(after_agg)

crop_type,mean_ndvi_after,n_parcels
sugarcane,0.336103448563994,19
wheat,0.3101333336697684,2
soybean,0.3126413049581258,4


In [0]:
# Final analytical output
result_df = before_agg.join(
    after_agg,
    "crop_type",
    "inner"
)

In [0]:
# formatting result df

result_df = result_df.select(
    "crop_type",
    F.round("mean_ndvi_before", 3).alias("mean_ndvi_before"),
    F.round("mean_ndvi_after", 3).alias("mean_ndvi_after"),
    "n_parcels"
)


display(result_df.limit(5))

crop_type,mean_ndvi_before,mean_ndvi_after,n_parcels
sugarcane,0.177,0.336,19
soybean,0.171,0.313,4
wheat,0.176,0.31,2


In [0]:
#Save analytical output


result_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        f"`{catalog_name}`.gold.gold_crop_ndvi_analysis"
    )


In [0]:
# save output in csv

result_df.write.mode("overwrite").option("header", True).csv(f"/Volumes/{catalog_name}/gold/csv_exports/gold_crop_ndvi_analysis_csv")